In [4]:

import pyspark
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
spark.version

In [7]:
import requests
from tqdm import tqdm  # uv add tqdm for progress bar

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"

response = requests.get(url, stream=True)
response.raise_for_status()

total_size = int(response.headers.get('content-length', 0))

with open("yellow_tripdata_2025-11.parquet", "wb") as file, tqdm(
    desc="Downloading",
    total=total_size,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
) as pbar:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            file.write(chunk)
            pbar.update(len(chunk))

print("\n✅ Download complete!")




Downloading: 100%|██████████| 67.8M/67.8M [00:04<00:00, 14.9MB/s]


✅ Download complete!


In [8]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [10]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [12]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [6]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [13]:
df = df.repartition(4)

In [15]:
df.write.parquet('yellow/2025/11/')

In [16]:
df = spark.read.parquet('yellow/2025/11/')

In [17]:
df.printSchema() 

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [18]:
!ls -lh yellow_tripdata_2025-11.parquet

-rw-r--r-- 1 jovyan users 68M Mar  9 10:34 yellow_tripdata_2025-11.parquet


In [19]:
!ls -lh yellow/2025/11/


total 98M
-rw-r--r-- 1 jovyan users 25M Mar  9 10:38 part-00000-cd13b5d4-0502-492f-9ea0-b2519d9595ec-c000.snappy.parquet
-rw-r--r-- 1 jovyan users 25M Mar  9 10:38 part-00001-cd13b5d4-0502-492f-9ea0-b2519d9595ec-c000.snappy.parquet
-rw-r--r-- 1 jovyan users 25M Mar  9 10:38 part-00002-cd13b5d4-0502-492f-9ea0-b2519d9595ec-c000.snappy.parquet
-rw-r--r-- 1 jovyan users 25M Mar  9 10:38 part-00003-cd13b5d4-0502-492f-9ea0-b2519d9595ec-c000.snappy.parquet
-rw-r--r-- 1 jovyan users   0 Mar  9 10:38 _SUCCESS


In [20]:
from pyspark.sql.functions import to_timestamp, col

df_filtered = df.filter(
    to_timestamp(col("tpep_pickup_datetime"), "yyyy-MM-dd HH:mm:ss") >= "2025-11-15 00:00:00"
).filter(
    to_timestamp(col("tpep_pickup_datetime"), "yyyy-MM-dd HH:mm:ss") < "2025-11-16 00:00:00"
)

trip_count = df_filtered.count()
print(f"Taxi trips started on 2025-11-15: {trip_count:,}")


Taxi trips started on 2025-11-15: 162,604


In [21]:
from pyspark.sql.functions import unix_timestamp, col, round

df = df.withColumn(
    "total_trip_time_minutes",
    round(
        (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60.0
    )
)


In [22]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|total_trip_time_minutes|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------------+
|       2| 2025-11-07 18:37:45|  2025-11-07 18:41:51|     

In [23]:
from pyspark.sql.functions import unix_timestamp, col, hour, max

df_longest = df.filter(
    col("total_trip_time_minutes") > 0  # Exclude invalid trips
).agg(
    max((unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600.0).alias("max_trip_hours")
).collect()[0]["max_trip_hours"]

print(f"Longest trip: {df_longest:.2f} hours")


Longest trip: 90.65 hours


In [24]:
zone_df = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

zone_df.createOrReplaceTempView("zones")


In [26]:
zone_df.show(20, truncate=False)  # Full column names

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park           |Boro Zone   |
|11        |Brooklyn     |Bath Beach             |Boro Zone   |
|12        |Manhattan    |Battery Park           |Yellow Zone |
|13        |Manhattan    |Battery Park C

In [28]:
df.createOrReplaceTempView("yellow_taxi")  # Name it

In [32]:
spark.sql("""
SELECT zones.Zone, COUNT(*) as trips
FROM yellow_taxi 
LEFT JOIN zones ON yellow_taxi.PULocationID = zones.LocationID
WHERE zones.Zone IS NOT NULL  -- Exclude NULL zones
GROUP BY zones.Zone
ORDER BY trips ASC
LIMIT 10
""").show()

+--------------------+-----+
|                Zone|trips|
+--------------------+-----+
|       Arden Heights|    1|
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Port Richmond|    3|
| Green-Wood Cemetery|    4|
|       Rikers Island|    4|
|         Great Kills|    4|
|   Rossville/Woodrow|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
+--------------------+-----+

